
# GVH Diagonal Cubic 0.3.2.5 — Explicit Static Radial Field Equations

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.5  
**Partie :** `0.2C2_weak_field_predictions`  
**Statut :** explicit radial reduction of the **candidate** timelike/vector action family

---

## Objective

The previous stage established the static spherical ansatz but deliberately left the radial field equations unspecified.

This notebook now derives a symmetry-reduced candidate system for

\[
ds^2=-A(r)\,dt^2+B(r)\,dr^2+r^2d\Omega^2,
\]

with the aligned unit-timelike branch

\[
u^\mu=(A^{-1/2},0,0,0).
\]

The scientific objectives are:

1. compute the exact geometric quantities for the static spherical metric;
2. compute the candidate vector invariants on the aligned branch;
3. derive the symmetry-reduced radial Euler–Lagrange equations;
4. verify that the GR vacuum equations are recovered when the candidate coupling vanishes;
5. perform a controlled weak-field expansion;
6. determine the local rank of the equations for the weak-field coefficients;
7. transform to isotropic coordinates before interpreting the PPN interfaces.

**Critical limitation:** the vector/timelike action remains a candidate ansatz introduced in 0.3.2.2. Results in this notebook are therefore conditional on that ansatz and on the aligned branch.



## 0. Scientific safeguards

This notebook must not:

- fit \(c_i\) to observations;
- call the candidate action an established GVH law;
- identify an areal-radius coefficient directly with a PPN coefficient without the coordinate transformation;
- declare a unique GVH prediction merely because the radial equations are solvable.

The key distinction is

\[
\text{conditional prediction of candidate family}
\neq
\text{unique prediction of GVH}.
\]


In [1]:

from __future__ import annotations

from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import sympy as sp

NOTEBOOK_ID = "GVH_Diagonal_Cubic_0.3.2.5"
VERSION = "0.3.2.5"

print(NOTEBOOK_ID, VERSION)
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH_Diagonal_Cubic_0.3.2.5 0.3.2.5
Python: 3.12.13
SymPy: 1.14.0



# 1. Exact static spherical geometry

Use coordinates

\[
x^\mu=(t,r,\theta,\phi)
\]

and metric

\[
g_{\mu\nu}
=
\operatorname{diag}
\left(
-A(r),\,B(r),\,r^2,\,r^2\sin^2\theta
\right).
\]

We compute the Christoffel symbols, Ricci tensor, Ricci scalar and Einstein tensor directly.


In [2]:

t, r, th, ph = sp.symbols("t r theta phi", real=True)
A = sp.Function("A")(r)
B = sp.Function("B")(r)

coords = [t, r, th, ph]
g = sp.diag(-A, B, r**2, r**2*sp.sin(th)**2)
g_inv = sp.simplify(g.inv())
n = 4

Gamma = [[[sp.Integer(0) for _ in range(n)] for _ in range(n)] for _ in range(n)]

for rho in range(n):
    for mu in range(n):
        for nu in range(n):
            expr = 0
            for sig in range(n):
                expr += g_inv[rho, sig] * (
                    sp.diff(g[sig, nu], coords[mu])
                    + sp.diff(g[sig, mu], coords[nu])
                    - sp.diff(g[mu, nu], coords[sig])
                )
            Gamma[rho][mu][nu] = sp.simplify(expr / 2)

nonzero_gamma = []
for rho in range(n):
    for mu in range(n):
        for nu in range(n):
            if Gamma[rho][mu][nu] != 0:
                nonzero_gamma.append((rho, mu, nu, Gamma[rho][mu][nu]))

print("Nonzero Christoffel entries:", len(nonzero_gamma))
nonzero_gamma[:12]


Nonzero Christoffel entries: 13


[(0, 0, 1, Derivative(A(r), r)/(2*A(r))),
 (0, 1, 0, Derivative(A(r), r)/(2*A(r))),
 (1, 0, 0, Derivative(A(r), r)/(2*B(r))),
 (1, 1, 1, Derivative(B(r), r)/(2*B(r))),
 (1, 2, 2, -r/B(r)),
 (1, 3, 3, -r*sin(theta)**2/B(r)),
 (2, 1, 2, 1/r),
 (2, 2, 1, 1/r),
 (2, 3, 3, -sin(2*theta)/2),
 (3, 1, 3, 1/r),
 (3, 2, 3, 1/tan(theta)),
 (3, 3, 1, 1/r)]

In [3]:

Ricci = [[sp.Integer(0) for _ in range(n)] for _ in range(n)]

for mu in range(n):
    for nu in range(n):
        expr = 0
        for rho in range(n):
            expr += sp.diff(Gamma[rho][mu][nu], coords[rho])
            expr -= sp.diff(Gamma[rho][mu][rho], coords[nu])
            for sig in range(n):
                expr += Gamma[rho][rho][sig] * Gamma[sig][mu][nu]
                expr -= Gamma[rho][nu][sig] * Gamma[sig][mu][rho]
        Ricci[mu][nu] = sp.simplify(expr)

R_scalar = sp.simplify(
    sum(g_inv[mu,nu] * Ricci[mu][nu] for mu in range(n) for nu in range(n))
)

Einstein = [
    [sp.simplify(Ricci[mu][nu] - g[mu,nu]*R_scalar/2) for nu in range(n)]
    for mu in range(n)
]

Gtt = sp.factor(Einstein[0][0])
Grr = sp.factor(Einstein[1][1])

print("G_tt =")
sp.pprint(Gtt)
print("\nG_rr =")
sp.pprint(Grr)


G_tt =
⎛  d           2          ⎞     
⎜r⋅──(B(r)) + B (r) - B(r)⎟⋅A(r)
⎝  dr                     ⎠     
────────────────────────────────
             2  2               
            r ⋅B (r)            

G_rr =
  d                          
r⋅──(A(r)) - A(r)⋅B(r) + A(r)
  dr                         
─────────────────────────────
            2                
           r ⋅A(r)           



For reference, the exact covariant components obtained computationally are

\[
G_{tt}
=
\frac{
A\,[rB'+B^2-B]
}{
r^2B^2
},
\]

\[
G_{rr}
=
\frac{
rA'+(1-B)A
}{
r^2A
}.
\]

These formulas provide a direct GR control later.


In [4]:

Gtt_expected = A * (r*sp.diff(B,r) + B**2 - B) / (r**2 * B**2)
Grr_expected = (r*sp.diff(A,r) + (1-B)*A) / (r**2 * A)

assert sp.simplify(Gtt - Gtt_expected) == 0
assert sp.simplify(Grr - Grr_expected) == 0

print("PASS — exact G_tt and G_rr formulas verified.")


PASS — exact G_tt and G_rr formulas verified.



# 2. Aligned unit-timelike field

Take

\[
u^\mu=(A^{-1/2},0,0,0).
\]

The normalization is exact:

\[
u^\mu u_\mu=-1.
\]

We now compute \(\nabla_\mu u_\nu\) directly from the same connection.


In [5]:

u_con = sp.Matrix([1/sp.sqrt(A), 0, 0, 0])
u_cov = sp.simplify(g * u_con)

norm = sp.simplify((u_con.T * u_cov)[0])
assert norm == -1

nabla_u_cov = [[sp.Integer(0) for _ in range(n)] for _ in range(n)]

for mu in range(n):
    for nu in range(n):
        expr = sp.diff(u_cov[nu], coords[mu])
        for rho in range(n):
            expr -= Gamma[rho][mu][nu] * u_cov[rho]
        nabla_u_cov[mu][nu] = sp.simplify(expr)

nonzero_nabla = [
    (mu,nu,nabla_u_cov[mu][nu])
    for mu in range(n) for nu in range(n)
    if nabla_u_cov[mu][nu] != 0
]

print("u.u =", norm)
print("Nonzero nabla_mu u_nu:")
nonzero_nabla


u.u = -1
Nonzero nabla_mu u_nu:


[(0, 1, Derivative(A(r), r)/(2*sqrt(A(r))))]


On this aligned static branch the only nonzero covariant derivative component is

\[
\nabla_t u_r=\frac{A'}{2\sqrt A}.
\]

This immediately simplifies the four candidate kinetic invariants.


In [6]:

# I1 = (nabla_mu u_nu)(nabla^mu u^nu)
I1 = 0
for mu in range(n):
    for nu in range(n):
        for a in range(n):
            for b in range(n):
                I1 += (
                    g_inv[mu,a] * g_inv[nu,b]
                    * nabla_u_cov[mu][nu]
                    * nabla_u_cov[a][b]
                )
I1 = sp.simplify(I1)

# Expansion theta = nabla_mu u^mu
# Compute nabla_mu u^nu
nabla_u_con = [[sp.Integer(0) for _ in range(n)] for _ in range(n)]
for mu in range(n):
    for nu in range(n):
        expr = sp.diff(u_con[nu], coords[mu])
        for rho in range(n):
            expr += Gamma[nu][mu][rho] * u_con[rho]
        nabla_u_con[mu][nu] = sp.simplify(expr)

theta_exp = sp.simplify(sum(nabla_u_con[mu][mu] for mu in range(n)))

# I3 = (nabla_mu u_nu)(nabla^nu u^mu)
I3 = 0
for mu in range(n):
    for nu in range(n):
        I3 += nabla_u_cov[mu][nu] * nabla_u_con[nu][mu]
I3 = sp.simplify(I3)

# acceleration a_mu = u^nu nabla_nu u_mu
a_cov = sp.Matrix([
    sp.simplify(sum(u_con[nu]*nabla_u_cov[nu][mu] for nu in range(n)))
    for mu in range(n)
])
a2 = sp.simplify((a_cov.T * g_inv * a_cov)[0])

print("I1 =", I1)
print("theta =", theta_exp)
print("I3 =", I3)
print("a^2 =", a2)


I1 = -Derivative(A(r), r)**2/(4*A(r)**2*B(r))
theta = 0
I3 = 0
a^2 = Derivative(A(r), r)**2/(4*A(r)**2*B(r))



The aligned branch gives

\[
I_1
=
-\frac{A'^2}{4A^2B},
\qquad
(\nabla_\mu u^\mu)^2=0,
\qquad
I_3=0,
\qquad
a_\mu a^\mu
=
\frac{A'^2}{4A^2B}.
\]

Therefore the vector kinetic sector

\[
\mathcal L_u
=
-c_1 I_1
-c_2(\nabla_\mu u^\mu)^2
-c_3 I_3
+c_4 a_\mu a^\mu
\]

collapses to

\[
\boxed{
\mathcal L_u
=
(c_1+c_4)\frac{A'^2}{4A^2B}
}.
\]

Define

\[
c_{14}=c_1+c_4.
\]

**Important result:** \(c_2\) and \(c_3\) drop out of this aligned static branch. Thus this branch cannot determine all four candidate couplings.


In [7]:

c1, c2, c3, c4, c14 = sp.symbols("c1 c2 c3 c4 c14", real=True)

L_u = sp.simplify(-c1*I1 - c2*theta_exp**2 - c3*I3 + c4*a2)
L_u_c14 = sp.simplify(L_u.subs(c1, c14-c4))

expected_Lu = sp.simplify(c14 * sp.diff(A,r)**2 / (4*A**2*B))

assert sp.simplify(L_u_c14 - expected_Lu) == 0

print("PASS — aligned branch depends only on c14 = c1 + c4.")
sp.pprint(expected_Lu)


PASS — aligned branch depends only on c14 = c1 + c4.
              2
    ⎛d       ⎞ 
c₁₄⋅⎜──(A(r))⎟ 
    ⎝dr      ⎠ 
───────────────
    2          
 4⋅A (r)⋅B(r)  



# 3. Symmetry-reduced radial action

Ignoring the common angular/time integration factor, use

\[
L_{\rm rad}
=
r^2\sqrt{AB}
\left[
R
+
c_{14}\frac{A'^2}{4A^2B}
\right].
\]

Because \(R\) contains \(A''\), the Euler–Lagrange equation for \(A\) must use the higher-derivative form

\[
\frac{\partial L}{\partial A}
-
\frac{d}{dr}\frac{\partial L}{\partial A'}
+
\frac{d^2}{dr^2}\frac{\partial L}{\partial A''}
=0.
\]

This is a **symmetry-reduced action calculation**. It is conditional on the candidate action and should later be cross-checked against the full tensor equations.


In [8]:

L_grav = sp.simplify(r**2 * sp.sqrt(A*B) * R_scalar)
L_vec = sp.simplify(r**2 * sp.sqrt(A*B) * expected_Lu)
L_rad = sp.simplify(L_grav + L_vec)

def euler_lagrange_radial(L, f, max_order=2):
    result = sp.diff(L, f)
    for k in range(1, max_order+1):
        fk = sp.diff(f, r, k)
        term = sp.diff(L, fk)
        if term != 0:
            result += (-1)**k * sp.diff(term, r, k)
    return sp.factor(sp.simplify(result))

EL_A = euler_lagrange_radial(L_rad, A, max_order=2)
EL_B = euler_lagrange_radial(L_rad, B, max_order=1)

print("Radial Euler-Lagrange equations derived.")


Radial Euler-Lagrange equations derived.



After removing nonzero common prefactors, the two independent reduced radial equations can be written as

\[
\boxed{
E_A=0
}
\]

with

\[
E_A=
-\frac{c_{14}}{2}r^2AB A''
+\frac{c_{14}}{4}r^2A A'B'
+\frac{3c_{14}}{8}r^2B A'^2
-c_{14}rAB A'
+rA^2B'
+A^2B^2-A^2B,
\]

and

\[
\boxed{
E_B=0
}
\]

with

\[
E_B=
-\frac{c_{14}}{8}r^2A'^2
-rAA'
+A^2B-A^2.
\]


In [9]:

EA_num = sp.factor(
    -c14*r**2*A*B*sp.diff(A,r,2)/2
    + c14*r**2*A*sp.diff(A,r)*sp.diff(B,r)/4
    + 3*c14*r**2*B*sp.diff(A,r)**2/8
    - c14*r*A*B*sp.diff(A,r)
    + r*A**2*sp.diff(B,r)
    + A**2*B**2
    - A**2*B
)

EB_num = sp.factor(
    -c14*r**2*sp.diff(A,r)**2/8
    - r*A*sp.diff(A,r)
    + A**2*B
    - A**2
)

# Compare EL equations up to nonzero multiplicative factors:
ratio_A = sp.simplify(EL_A / EA_num)
ratio_B = sp.simplify(EL_B / EB_num)

print("EL_A / E_A =", ratio_A)
print("EL_B / E_B =", ratio_B)

assert sp.simplify(EL_A - ratio_A*EA_num) == 0
assert sp.simplify(EL_B - ratio_B*EB_num) == 0

print("PASS — explicit radial equations match the reduced action variation.")


EL_A / E_A = (A(r)*B(r))**(3/2)/(A(r)**4*B(r)**3)
EL_B / E_B = (A(r)*B(r))**(3/2)/(A(r)**3*B(r)**3)
PASS — explicit radial equations match the reduced action variation.



# 4. GR control

Set

\[
c_{14}=0.
\]

Then

\[
E_A=0
\quad\Rightarrow\quad
rB'+B^2-B=0,
\]

and

\[
E_B=0
\quad\Rightarrow\quad
rA'+(1-B)A=0.
\]

These are precisely equivalent to the vacuum conditions

\[
G_{tt}=0,
\qquad
G_{rr}=0
\]

for the static spherical metric.


In [10]:

EA_GR = sp.factor(EA_num.subs(c14,0) / A**2)
EB_GR = sp.factor(EB_num.subs(c14,0) / A)

EA_GR_expected = sp.factor(r*sp.diff(B,r) + B**2 - B)
EB_GR_expected = sp.factor(-r*sp.diff(A,r) + A*B - A)

assert sp.simplify(EA_GR - EA_GR_expected) == 0
assert sp.simplify(EB_GR - EB_GR_expected) == 0

# Compare directly with Einstein equations:
assert sp.simplify(EA_GR_expected - (r**2*B**2/A)*Gtt) == 0
assert sp.simplify(EB_GR_expected + r**2*A*Grr) == 0

print("PASS — c14 -> 0 reproduces the exact GR vacuum radial equations.")


PASS — c14 -> 0 reproduces the exact GR vacuum radial equations.



# 5. Weak-field expansion in areal radius

Introduce

\[
x=\frac{m}{r},
\]

and expand

\[
A(r)
=
1-2a_1x+2a_2^{(R)}x^2+\mathcal O(x^3),
\]

\[
B(r)
=
1+2b_1^{(R)}x+b_2^{(R)}x^2+\mathcal O(x^3).
\]

The superscript \((R)\) emphasizes **areal-radius coordinates**.

This distinction is essential because the PPN definitions used previously are expressed in isotropic spatial coordinates.


In [11]:

m, xeps = sp.symbols("m x", positive=True)
a1w, a2R, b1R, b2R = sp.symbols("a1 a2R b1R b2R", real=True)

x = m/r
A_series = 1 - 2*a1w*x + 2*a2R*x**2
B_series = 1 + 2*b1R*x + b2R*x**2

EA_series_raw = sp.expand(
    EA_num.subs({A:A_series, B:B_series}).doit()
)
EB_series_raw = sp.expand(
    EB_num.subs({A:A_series, B:B_series}).doit()
)

# Rewrite using xeps = m/r by setting r = m/xeps
EA_x = sp.series(
    sp.simplify(EA_series_raw.subs(r, m/xeps)),
    xeps, 0, 4
).removeO().expand()

EB_x = sp.series(
    sp.simplify(EB_series_raw.subs(r, m/xeps)),
    xeps, 0, 4
).removeO().expand()

print("E_A series:")
sp.pprint(sp.collect(EA_x, xeps))
print("\nE_B series:")
sp.pprint(sp.collect(EB_x, xeps))


E_A series:
                                                                               ↪
 3 ⎛    2                                   2                                  ↪
x ⋅⎝5⋅a₁ ⋅b1R⋅c₁₄ - 2⋅a₁⋅a2R⋅c₁₄ - 16⋅a₁⋅b1R  - a₁⋅b2R⋅c₁₄ + 4⋅a₁⋅b2R - 2⋅a2R⋅ ↪
                                                                               ↪

↪                           ⎛    2                                             ↪
↪                    ⎞    2 ⎜3⋅a₁ ⋅c₁₄                                 2       ↪
↪ b1R⋅c₁₄ + 4⋅b1R⋅b2R⎠ + x ⋅⎜───────── - a₁⋅b1R⋅c₁₄ - 2⋅a2R⋅c₁₄ + 4⋅b1R  - b2R ↪
↪                           ⎝    2                                             ↪

↪ ⎞
↪ ⎟
↪ ⎟
↪ ⎠

E_B series:
                                                                      ⎛    2   ↪
 3 ⎛    2                                                      ⎞    2 ⎜  a₁ ⋅c ↪
x ⋅⎝8⋅a₁ ⋅b1R + 2⋅a₁⋅a2R⋅c₁₄ - 12⋅a₁⋅a2R - 4⋅a₁⋅b2R + 8⋅a2R⋅b1R⎠ + x ⋅⎜- ───── ↪
                                                                  


The leading coefficient equations are extracted order by order.

At \(\mathcal O(x)\), \(E_B=0\) yields a relation between the leading temporal and spatial potentials.

At \(\mathcal O(x^2)\), the two equations determine the next areal-radius coefficients conditionally on \(a_1\) and \(c_{14}\).


In [12]:

eq_B1 = sp.expand(EB_x).coeff(xeps,1)
eq_A2 = sp.expand(EA_x).coeff(xeps,2)
eq_B2 = sp.expand(EB_x).coeff(xeps,2)

weak_equations = pd.DataFrame([
    {"order":"B @ x^1", "equation":str(sp.factor(eq_B1))},
    {"order":"A @ x^2", "equation":str(sp.factor(eq_A2))},
    {"order":"B @ x^2", "equation":str(sp.factor(eq_B2))},
])

weak_equations


,order,equation
0,B @ x^1,-2*(a1 - b1R)
1,A @ x^2,(3*a1**2*c14 - 2*a1*b1R*c14 - 4*a2R*c14 + 8*b1...
2,B @ x^2,-(a1**2*c14 - 8*a1**2 + 16*a1*b1R - 8*a2R - 2*...


In [13]:

weak_solution = sp.solve(
    [eq_B1, eq_A2, eq_B2],
    [b1R, a2R, b2R],
    dict=True
)

assert len(weak_solution) == 1
weak_solution


[{a2R: 0, b1R: a1, b2R: a1**2*(c14 + 8)/2}]


The symmetry-reduced candidate equations give

\[
\boxed{
b_1^{(R)}=a_1
}
\]

\[
\boxed{
a_2^{(R)}=0
}
\]

\[
\boxed{
b_2^{(R)}
=
\frac{a_1^2}{2}(c_{14}+8)
}.
\]

This is a substantive conditional result: the candidate coupling first appears at second order in the areal-radius spatial metric.

However, **\(a_2^{(R)}=0\) must not be inserted directly into the earlier PPN formula for \(\beta\)**, because that formula uses isotropic coordinates.



# 6. Rank audit

We now compute the Jacobian of the three leading weak-field equations with respect to

\[
(b_1^{(R)},a_2^{(R)},b_2^{(R)}).
\]

This tests whether those coefficients are locally determined for fixed \(a_1\) and \(c_{14}\).


In [14]:

weak_unknowns = [b1R, a2R, b2R]
weak_eqs = [eq_B1, eq_A2, eq_B2]

J = sp.Matrix(weak_eqs).jacobian(weak_unknowns)
J_on_solution = sp.simplify(J.subs(weak_solution[0]))
rank_weak = J_on_solution.rank()

print("Jacobian:")
sp.pprint(J_on_solution)
print("Rank =", rank_weak)

assert rank_weak == 3


Jacobian:
⎡     2          0     0 ⎤
⎢                        ⎥
⎢a₁⋅(8 - c₁₄)  -2⋅c₁₄  -1⎥
⎢                        ⎥
⎣   -8⋅a₁        4     1 ⎦
Rank = 3



Thus

\[
\boxed{\operatorname{rank}J=3}.
\]

For fixed \(a_1\) and \(c_{14}\), this static aligned candidate branch locally determines the three displayed areal-radius coefficients.

This **does not** fix \(c_{14}\), nor does it select the candidate action as uniquely GVH.



# 7. Transform to isotropic coordinates

Introduce an isotropic radius \(\rho\) and

\[
y=\frac{m}{\rho}.
\]

Write

\[
r
=
\rho
\left(
1+p_1 y+p_2y^2+\mathcal O(y^3)
\right).
\]

The spatial metric must satisfy

\[
B(r)\,dr^2+r^2d\Omega^2
=
C(\rho)\left(d\rho^2+\rho^2d\Omega^2\right).
\]

The angular sector gives

\[
C(\rho)=\frac{r^2}{\rho^2}.
\]

The radial sector then fixes \(p_1,p_2\).


In [15]:

y, p1, p2 = sp.symbols("y p1 p2", real=True)

sol_areal = weak_solution[0]

x_of_y = sp.series(
    y/(1 + p1*y + p2*y**2),
    y, 0, 3
).removeO()

B_y = sp.expand(
    (1 + 2*b1R*x_of_y + b2R*x_of_y**2).subs(sol_areal)
)

# r = rho + p1*m + p2*m^2/rho => dr/drho = 1 - p2*y^2
dr_drho = 1 - p2*y**2

lhs_radial = sp.series(B_y * dr_drho**2, y, 0, 3).removeO().expand()
C_iso = sp.series((1 + p1*y + p2*y**2)**2, y, 0, 3).removeO().expand()

eq_iso1 = sp.Eq(lhs_radial.coeff(y,1), C_iso.coeff(y,1))
eq_iso2 = sp.Eq(lhs_radial.coeff(y,2), C_iso.coeff(y,2))

iso_transform_solution = sp.solve(
    [eq_iso1, eq_iso2],
    [p1,p2],
    dict=True
)

assert len(iso_transform_solution) == 1
iso_transform_solution


[{p1: a1, p2: a1**2*(c14 + 2)/8}]


The isotropic coordinate transformation is

\[
\boxed{
p_1=a_1
}
\]

and

\[
\boxed{
p_2=\frac{a_1^2}{8}(c_{14}+2)
}.
\]


In [16]:

iso_sol = iso_transform_solution[0]

# Temporal metric in isotropic radius
x_y_fixed = sp.series(
    x_of_y.subs(iso_sol),
    y, 0, 3
).removeO()

A_iso = sp.series(
    (1 - 2*a1w*x_y_fixed + 2*a2R*x_y_fixed**2).subs(sol_areal),
    y, 0, 3
).removeO().expand()

C_iso_fixed = sp.series(
    C_iso.subs(iso_sol),
    y, 0, 3
).removeO().expand()

print("A_iso =")
sp.pprint(A_iso)
print("\nC_iso =")
sp.pprint(C_iso_fixed)


A_iso =
    2  2             
2⋅a₁ ⋅y  - 2⋅a₁⋅y + 1

C_iso =
  2      2       2  2             
a₁ ⋅c₁₄⋅y    3⋅a₁ ⋅y              
────────── + ──────── + 2⋅a₁⋅y + 1
    4           2                 



The isotropic expansion becomes

\[
A_{\rm iso}
=
1-2a_1y+2a_1^2y^2+\mathcal O(y^3),
\]

so

\[
g_{tt}
=
-1+2a_1y-2a_1^2y^2+\cdots.
\]

The spatial conformal factor starts as

\[
C(\rho)
=
1+2a_1y+\mathcal O(y^2).
\]

Therefore, after Newtonian normalization of the leading potential, the familiar PPN interfaces satisfy conditionally on this aligned candidate branch

\[
\boxed{\gamma=1},
\qquad
\boxed{\beta=1},
\]

independently of \(c_{14}\) at this order.

This is a **candidate-family result**, not yet a unique GVH prediction.


In [17]:

# Extract isotropic PPN-like coefficients
Aiso_y1 = sp.expand(A_iso).coeff(y,1)
Aiso_y2 = sp.expand(A_iso).coeff(y,2)
Ciso_y1 = sp.expand(C_iso_fixed).coeff(y,1)

a1_iso = sp.simplify(-Aiso_y1/2)
a2_iso = sp.simplify(Aiso_y2/2)
b1_iso = sp.simplify(Ciso_y1/2)

gamma_candidate = sp.simplify(b1_iso/a1_iso)
beta_candidate = sp.simplify(a2_iso/a1_iso**2)

print("a1_iso =", a1_iso)
print("a2_iso =", a2_iso)
print("b1_iso =", b1_iso)
print("gamma_candidate =", gamma_candidate)
print("beta_candidate =", beta_candidate)

assert gamma_candidate == 1
assert beta_candidate == 1


a1_iso = a1
a2_iso = a1**2
b1_iso = a1
gamma_candidate = 1
beta_candidate = 1



# 8. Interpretation of the \(c_{14}\) dependence

The coupling \(c_{14}\) survives in the second-order **spatial** isotropic metric through the coefficient of \(y^2\), even though

\[
\gamma=\beta=1
\]

at the standard first PPN interfaces derived above.

Therefore this branch does **not** imply that the candidate vector sector is observationally invisible.

It means only that the two interfaces \(\gamma,\beta\) do not constrain \(c_{14}\) at this order.

Potentially informative sectors may instead include:

- higher-order spatial post-Newtonian structure;
- preferred-frame PPN parameters;
- non-aligned vector configurations;
- time-dependent systems;
- strong-field observables.

Those must be derived before comparison with data.


In [18]:

c14_visibility = pd.DataFrame([
    {"observable/interface":"gamma", "depends_on_c14_here":False, "status":"CANDIDATE_BRANCH_RESULT"},
    {"observable/interface":"beta", "depends_on_c14_here":False, "status":"CANDIDATE_BRANCH_RESULT"},
    {"observable/interface":"second-order isotropic spatial coefficient", "depends_on_c14_here":True, "status":"DERIVED_CANDIDATE_FAMILY"},
    {"observable/interface":"preferred-frame parameters", "depends_on_c14_here":"NOT_DERIVED", "status":"NEXT_THEORY_TARGET"},
])
c14_visibility


,observable/interface,depends_on_c14_here,status
0,gamma,False,CANDIDATE_BRANCH_RESULT
1,beta,False,CANDIDATE_BRANCH_RESULT
2,second-order isotropic spatial coefficient,True,DERIVED_CANDIDATE_FAMILY
3,preferred-frame parameters,NOT_DERIVED,NEXT_THEORY_TARGET



# 9. What has and has not been closed

### Closed conditionally on the candidate action + aligned branch

For fixed \(a_1\) and \(c_{14}\), the weak-field radial equations have rank 3 and determine

\[
b_1^{(R)},\ a_2^{(R)},\ b_2^{(R)}.
\]

### Still not closed as a unique GVH theory

The following remain unresolved:

- why this vector action is the unique GVH action;
- why the aligned branch is physically selected;
- the theoretical value of \(c_{14}\);
- the roles of \(c_2,c_3\) outside this branch;
- perturbative stability and ghost constraints;
- preferred-frame observables;
- full independent tensor-variation cross-check.


In [19]:

closure_audit = pd.DataFrame([
    {"item":"explicit reduced radial equations", "status":"DERIVED_CONDITIONAL_ON_CANDIDATE_ACTION"},
    {"item":"weak-field local coefficient rank", "status":"3_FOR_FIXED_a1_c14"},
    {"item":"gamma", "status":"1_IN_ALIGNED_CANDIDATE_BRANCH"},
    {"item":"beta", "status":"1_IN_ALIGNED_CANDIDATE_BRANCH"},
    {"item":"c14", "status":"UNFIXED"},
    {"item":"c2,c3 in aligned static branch", "status":"DROP_OUT"},
    {"item":"unique GVH action", "status":"NOT_DERIVED"},
    {"item":"unique GVH prediction", "status":"BLOCKED"},
])
closure_audit


,item,status
0,explicit reduced radial equations,DERIVED_CONDITIONAL_ON_CANDIDATE_ACTION
1,weak-field local coefficient rank,3_FOR_FIXED_a1_c14
2,gamma,1_IN_ALIGNED_CANDIDATE_BRANCH
3,beta,1_IN_ALIGNED_CANDIDATE_BRANCH
4,c14,UNFIXED
5,"c2,c3 in aligned static branch",DROP_OUT
6,unique GVH action,NOT_DERIVED
7,unique GVH prediction,BLOCKED



# 10. Final scientific status


In [20]:

internal_checks = {
    "unit_timelike_normalization": norm == -1,
    "aligned_Lu_only_c14": sp.simplify(L_u_c14 - expected_Lu) == 0,
    "radial_EL_A_verified": sp.simplify(EL_A - ratio_A*EA_num) == 0,
    "radial_EL_B_verified": sp.simplify(EL_B - ratio_B*EB_num) == 0,
    "GR_limit_verified": True,
    "weak_field_rank_is_3": rank_weak == 3,
    "isotropic_gamma_is_1": gamma_candidate == 1,
    "isotropic_beta_is_1": beta_candidate == 1,
    "c14_remains_unfixed": True,
    "no_observational_data_used": True,
}

for key,value in internal_checks.items():
    print(f"{key}: {value}")

assert all(internal_checks.values())

FINAL_STATUS = (
    "PASS-EXPLICIT-RADIAL-EQUATIONS_RANK-3-CONDITIONAL-ON-c14_"
    "PPN-GAMMA-BETA-GR-LIKE_BLOCKED-UNIQUE-GVH-PREDICTION"
)

print("\nFINAL STATUS:", FINAL_STATUS)


unit_timelike_normalization: True
aligned_Lu_only_c14: True
radial_EL_A_verified: True
radial_EL_B_verified: True
GR_limit_verified: True
weak_field_rank_is_3: True
isotropic_gamma_is_1: True
isotropic_beta_is_1: True
c14_remains_unfixed: True
no_observational_data_used: True

FINAL STATUS: PASS-EXPLICIT-RADIAL-EQUATIONS_RANK-3-CONDITIONAL-ON-c14_PPN-GAMMA-BETA-GR-LIKE_BLOCKED-UNIQUE-GVH-PREDICTION



# 11. Machine-readable artifact


In [21]:

artifact = {
    "notebook": NOTEBOOK_ID,
    "version": VERSION,
    "final_status": FINAL_STATUS,
    "action_status": "CANDIDATE_ANSATZ",
    "branch": "static_spherical_aligned_unit_timelike",
    "effective_static_coupling": "c14 = c1 + c4",
    "c2_c3_status_in_this_branch": "DROP_OUT",
    "radial_equations_derived": True,
    "weak_field_rank": int(rank_weak),
    "areal_radius_solution": {
        "b1R": str(sp.simplify(sol_areal[b1R])),
        "a2R": str(sp.simplify(sol_areal[a2R])),
        "b2R": str(sp.simplify(sol_areal[b2R])),
    },
    "isotropic_interfaces": {
        "gamma_candidate": str(gamma_candidate),
        "beta_candidate": str(beta_candidate),
    },
    "c14_fixed_by_GVH": False,
    "unique_GVH_action_selected": False,
    "unique_GVH_prediction_ready": False,
    "observational_data_used": False,
    "next_theory_targets": [
        "independent_full_tensor_variation_cross_check",
        "coupling_selection_or_stability_constraints",
        "preferred_frame_PPN_sector",
        "non_aligned_or_time_dependent_branch"
    ],
}

if Path("/content").exists():
    EXPORT_DIR = Path("/content/gvh_exports")
else:
    EXPORT_DIR = Path.cwd() / "gvh_exports"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = EXPORT_DIR / "gvh_0.3.2.5_explicit_static_radial_field_equations.json"

artifact_path.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

assert artifact_path.exists()
assert artifact_path.stat().st_size > 0
assert artifact["observational_data_used"] is False
assert artifact["unique_GVH_prediction_ready"] is False
assert artifact["weak_field_rank"] == 3

print("PASS — 0.3.2.5 audit artifact created.")
print("Artifact:", artifact_path)
print("Status:", FINAL_STATUS)


PASS — 0.3.2.5 audit artifact created.
Artifact: /content/gvh_exports/gvh_0.3.2.5_explicit_static_radial_field_equations.json
Status: PASS-EXPLICIT-RADIAL-EQUATIONS_RANK-3-CONDITIONAL-ON-c14_PPN-GAMMA-BETA-GR-LIKE_BLOCKED-UNIQUE-GVH-PREDICTION



# Conclusion

This notebook obtains the first nonzero weak-field closure rank in the 0.3.2 chain.

For the candidate timelike/vector action and the static aligned branch,

\[
\mathcal L_u
=
c_{14}\frac{A'^2}{4A^2B},
\qquad
c_{14}=c_1+c_4,
\]

and the symmetry-reduced radial equations are explicitly derived.

Their weak-field expansion yields a rank-3 system for the displayed areal-radius coefficients, conditionally on \(a_1\) and \(c_{14}\).

After the necessary transformation to isotropic coordinates,

\[
\boxed{\gamma=1},
\qquad
\boxed{\beta=1}
\]

for this candidate branch at the standard PPN interfaces.

This does **not** make the model identical to GR and does **not** constitute a unique GVH prediction, because:

- \(c_{14}\) remains unfixed;
- \(c_2,c_3\) are invisible in this branch;
- higher-order spatial structure retains candidate-coupling dependence;
- the underlying action is not uniquely derived from GVH principles;
- stability and preferred-frame sectors remain unaudited.

Expected final status:

```text
PASS-EXPLICIT-RADIAL-EQUATIONS_RANK-3-CONDITIONAL-ON-c14_PPN-GAMMA-BETA-GR-LIKE_BLOCKED-UNIQUE-GVH-PREDICTION
```
